# nanochat-ru training notebook (Kaggle T4 x2)

Easiest way to run this: in Kaggle, New Notebook -> File -> Upload Notebook, and upload this
`.ipynb` file directly (no need to copy cells by hand). If you'd rather copy cells manually,
copy each **code cell** below in order; markdown cells are for reference only.

**Round 2 (`d6`)**: bigger architecture than the finished `d4` run (73.53M params vs 36.7M),
same Chinchilla-compute-optimal `--target-param-data-ratio=20` (~464.0M tokens). Chosen over
"overtraining" `d4` on more data (the originally-planned `d4v2`) after
`kaggle_vram_probe.ipynb` confirmed `d6` fits a T4 up to `--device-batch-size=13` -- using 8
here instead, since that's the value that evenly divides the auto-computed
`total_batch_size` (see Cell 1's comment; 12/13 don't) -- see `docs/RESEARCH_LOG.md` for the
full reasoning. Estimated ~4.3h pretrain on T4 x2. New model tag so it can't collide with /
accidentally resume into the finished `d4` checkpoint, which stays on Drive untouched as a
before/after comparison point.

Before running:
1. `REPO_URL` in Cell 1 already points at https://github.com/nadeko0/nanochat-ru.git
2. Set up Kaggle Secrets `GDRIVE_CLIENT_ID`, `GDRIVE_CLIENT_SECRET`, `GDRIVE_OAUTH_TOKEN`, `GDRIVE_FOLDER_ID` (four separate single-line secrets, not one multi-line blob -- Kaggle's Secrets box doesn't reliably keep newlines) -- see `docs/RCLONE_GDRIVE_SETUP.md`. Attach all four to the notebook (Add-ons -> Secrets) before running.
3. Enable a T4 x2 GPU accelerator in Notebook settings, and internet access.
4. `SMOKE_TEST = True` in Cell 1 by default -- first run end to end like this (~5-10 min) to confirm the whole pipeline works (clone, rclone auth, Drive round-trip, torchrun, checkpoint sync), before flipping it to `False` for a real multi-session training run.

Each session is capped at 12h and can be interrupted earlier. Checkpoints sync to Google Drive continuously (Cell 4) so a killed session loses at most one `--save-every` interval -- if a session dies partway through the ~4.3h pretrain, just start a fresh Kaggle session with this same notebook and Cell 2 will auto-detect the `d6` checkpoint on Drive and resume from the last saved step.

## Cell 1: clone repo, install dependencies, Rust toolchain, rclone

In [6]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/nadeko0/nanochat-ru.git"
REPO_DIR = "/kaggle/working/repo"
# "d4" (depth=4, ratio=20, 230.7M tokens) is already trained and finished -- see README.md.
# Round 2: instead of overtraining the same tiny d4 architecture on more data (the originally
# planned "d4v2"), a VRAM probe (kaggle/kaggle_vram_probe.ipynb, see docs/RESEARCH_LOG.md)
# confirmed depth=6 (73.53M params, ~2x d4) fits comfortably on a T4 at device-batch-size=13.
# Going wider/deeper at the same Chinchilla-compute-optimal ratio=20 is more principled than
# overtraining a fixed small model, and cheaper in wall-clock too (~4.3h estimated vs ~5.3h).
DEPTH = 6
# base_train.py's --target-param-data-ratio auto-computes total_batch_size=262,144 tokens
# (tuned via muP-style scaling from a d12 reference, not something to override lightly).
# device_batch_size * max_seq_len(2048) * world_size(2) must divide that evenly, i.e.
# device_batch_size must divide 64 -- 12 or 13 (what the VRAM probe found) don't. 8 is the
# largest clean divisor of 64 at or below the probe's verified-safe ceiling of 13.
DEVICE_BATCH_SIZE = 8
MODEL_TAG = f"d{DEPTH}"

# True = ~20 optimizer steps on ~2 data shards, just to prove the whole pipeline (clone,
# rclone, Drive up/download, torchrun, checkpoint sync) actually works end to end, in a
# few minutes. Flip to False once a smoke-test run has gone green, for the real training run.
SMOKE_TEST = False

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("Repo already present, pulling latest...")
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)

def have(cmd):
    return subprocess.run(["bash", "-lc", f"command -v {cmd}"], capture_output=True).returncode == 0

# uv (fast Python package/dependency manager)
if not have("uv"):
    !curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = f"{os.path.expanduser('~/.local/bin')}:{os.environ['PATH']}"

# Rust toolchain (needed to build rustbpe, nanochat's tokenizer, from source)
if not have("cargo"):
    !curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
os.environ["PATH"] = f"{os.path.expanduser('~/.cargo/bin')}:{os.environ['PATH']}"

# rclone (for Google Drive checkpoint sync)
if not have("rclone"):
    !curl https://rclone.org/install.sh | sudo bash

# Install deps into the *kernel's own* Python (sys.executable), not an isolated .venv --
# `uv sync` creates a separate .venv that this notebook process/kernel never activates,
# so later `import rustbpe` etc. would fail even though the install "succeeded".
!uv pip install --system --python {sys.executable} --extra gpu -r pyproject.toml

print(f"Cell 1 done. SMOKE_TEST={SMOKE_TEST}")

Repo already present, pulling latest...
Already up to date.
Using Python 3.12.13 environment at: /usr
Resolved 84 packages in 65ms                                         
Checked 84 packages in 2ms
Cell 1 done. SMOKE_TEST=False


## Cell 2: configure rclone from Kaggle Secrets, pull existing checkpoint (if any)

Requires `GDRIVE_CLIENT_ID`, `GDRIVE_CLIENT_SECRET`, `GDRIVE_OAUTH_TOKEN`, `GDRIVE_FOLDER_ID` secrets attached to the notebook -- see `docs/RCLONE_GDRIVE_SETUP.md`. Four separate single-line secrets, not one multi-line blob: Kaggle's Secrets box is a single-line field, so a pasted multi-line `rclone.conf` snippet loses its newlines and gets mangled.

Uses a personal OAuth rclone remote, not a service account: on a personal (non-Workspace) Google
Drive, service accounts have no storage quota of their own and can't create files in a shared
folder (`storageQuotaExceeded`), even with Editor access -- that only works with Shared Drives,
which require Google Workspace. A personal OAuth token writes files as you, so it consumes your
own 5TB quota. Trade-off: it can access your whole Drive (`scope=drive`), not just one folder --
that's inherent to how personal Drive permissions work, not something scoping the token down can
fix here.

In [7]:
import os
import subprocess
import sys
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
# Four separate single-line secrets -- Kaggle's Secrets box doesn't reliably preserve
# newlines pasted into it, so a combined multi-line rclone.conf snippet gets mangled.
client_id = secrets.get_secret("GDRIVE_CLIENT_ID").strip()
client_secret = secrets.get_secret("GDRIVE_CLIENT_SECRET").strip()
oauth_token = secrets.get_secret("GDRIVE_OAUTH_TOKEN").strip()
folder_id = secrets.get_secret("GDRIVE_FOLDER_ID").strip()

rclone_conf_dir = os.path.expanduser("~/.config/rclone")
os.makedirs(rclone_conf_dir, exist_ok=True)
with open(os.path.join(rclone_conf_dir, "rclone.conf"), "w") as f:
    f.write(
        "[gdrive]\n"
        "type = drive\n"
        "scope = drive\n"
        f"client_id = {client_id}\n"
        f"client_secret = {client_secret}\n"
        f"token = {oauth_token}\n"
        f"root_folder_id = {folder_id}\n"
        "team_drive =\n"
    )

# sanity check -- should list existing subfolders (or nothing on a first run), not an auth error
!rclone lsd gdrive:

DRIVE_REMOTE = "gdrive:"
NANOCHAT_BASE_DIR = "/kaggle/working/nanochat_cache"
os.environ["NANOCHAT_BASE_DIR"] = NANOCHAT_BASE_DIR
os.makedirs(NANOCHAT_BASE_DIR, exist_ok=True)

# Smoke-test runs use a separate model tag so they never collide with (or get resumed
# into) the real run's checkpoints -- their --max-seq-len/--num-iterations differ, so
# sharing a tag would break resume with a shape/schedule mismatch.
EFFECTIVE_MODEL_TAG = f"{MODEL_TAG}-smoketest" if SMOKE_TEST else MODEL_TAG
os.environ["EFFECTIVE_MODEL_TAG"] = EFFECTIVE_MODEL_TAG

# Pull down anything already on Drive from a previous session: tokenizer + checkpoints
for subdir in ["tokenizer", "base_checkpoints", "chatsft_checkpoints"]:
    remote_path = f"{DRIVE_REMOTE}{subdir}"
    local_path = os.path.join(NANOCHAT_BASE_DIR, subdir)
    listing = subprocess.run(["rclone", "lsf", remote_path], capture_output=True, text=True)
    if listing.returncode == 0 and listing.stdout.strip():
        print(f"Found {subdir} on Drive, downloading...")
        !rclone copy {remote_path} {local_path} --checksum -v
    else:
        print(f"No {subdir} on Drive yet.")

# Figure out whether we can resume base pretraining from a prior session's checkpoint
# (only meaningful for real runs -- smoke tests always start fresh from a random init)
sys.path.insert(0, REPO_DIR)
from nanochat.checkpoint_manager import find_last_step

RESUME_STEP = -1
base_ckpt_dir = os.path.join(NANOCHAT_BASE_DIR, "base_checkpoints", EFFECTIVE_MODEL_TAG)
if not SMOKE_TEST and os.path.isdir(base_ckpt_dir):
    try:
        RESUME_STEP = find_last_step(base_ckpt_dir)
        print(f"Found existing base checkpoint at step {RESUME_STEP}, will resume from it.")
    except FileNotFoundError:
        print("base_checkpoints dir exists but has no checkpoints in it yet.")
else:
    print("No prior base checkpoint found (or SMOKE_TEST=True), starting fresh.")

os.environ["RESUME_STEP"] = str(RESUME_STEP)

           0 2026-08-10 16:16:45        -1 base_checkpoints
           0 2026-08-10 15:53:07        -1 base_data_climbmix
           0 2026-08-10 17:58:32        -1 chatsft_checkpoints
           0 2026-08-10 15:56:35        -1 tokenizer
Found tokenizer on Drive, downloading...
2026/08/10 18:56:00 INFO  : There was nothing to transfer
2026/08/10 18:56:00 INFO  : 
Transferred:   	          0 B / 0 B, -, 0 B/s, ETA -
Checks:                 2 / 2, 100%, Listed 4
Elapsed time:         0.3s

Found base_checkpoints on Drive, downloading...
2026/08/10 18:56:03 INFO  : There was nothing to transfer
2026/08/10 18:56:03 INFO  : 
Transferred:   	          0 B / 0 B, -, 0 B/s, ETA -
Checks:                20 / 20, 100%, Listed 42
Elapsed time:         1.6s

Found chatsft_checkpoints on Drive, downloading...
2026/08/10 18:56:05 INFO  : There was nothing to transfer
2026/08/10 18:56:05 INFO  : 
Transferred:   	          0 B / 0 B, -, 0 B/s, ETA -
Checks:                 4 / 4, 100%, Listed 10
Elaps

## Cell 3 (optional): reuse cached dataset from Drive instead of re-downloading

The pretraining corpus (ClimbMix parquet shards) is static -- once downloaded, cache it on Drive so future sessions skip the download entirely.

In [8]:
import os
import subprocess

DRIVE_REMOTE = "gdrive:"
NANOCHAT_BASE_DIR = os.environ["NANOCHAT_BASE_DIR"]
DATA_DIR = os.path.join(NANOCHAT_BASE_DIR, "base_data_climbmix")
DATA_REMOTE = f"{DRIVE_REMOTE}base_data_climbmix"
TOKENIZER_DIR = os.path.join(NANOCHAT_BASE_DIR, "tokenizer")
TOKENIZER_REMOTE = f"{DRIVE_REMOTE}tokenizer"

# --target-param-data-ratio=20 (Cell 4) resolves to ~464.0M training tokens for depth=6
# (measured for d4: ratio=20 -> 230.7M tokens from 20 shards, i.e. ~11.5M tokens/shard --
# that ratio is a property of the ClimbMix data + tokenizer, not the model, so it carries
# over). 45 shards gives some margin; base_train.py will log if it still wants more.
# SMOKE_TEST only needs 2 shards (1 train + 1 val) to prove the pipeline works.
NUM_SHARDS = 2 if SMOKE_TEST else 45

listing = subprocess.run(["rclone", "lsf", DATA_REMOTE], capture_output=True, text=True)
if listing.returncode == 0 and listing.stdout.strip():
    print("Found cached dataset on Drive, downloading instead of re-fetching from HuggingFace...")
    !rclone copy {DATA_REMOTE} {DATA_DIR} --checksum -v
else:
    print(f"No cached dataset on Drive yet, downloading {NUM_SHARDS} shards from source...")
    !python -m nanochat.dataset -n {NUM_SHARDS}
    if not SMOKE_TEST:
        print("Caching dataset to Drive for future sessions...")
        !rclone copy {DATA_DIR} {DATA_REMOTE} --checksum -v

# base_train.py requires a trained tokenizer (nanochat/tokenizer.py get_tokenizer() loads
# NANOCHAT_BASE_DIR/tokenizer/tokenizer.pkl). Cell 2 already pulled one down from Drive if a
# prior session made one -- only train a fresh one if that didn't happen. Same English/ClimbMix
# tokenizer as the original d4 run -- the tokenizer doesn't depend on model depth, no need to
# retrain it for d6.
tokenizer_pkl = os.path.join(TOKENIZER_DIR, "tokenizer.pkl")
if os.path.exists(tokenizer_pkl):
    print(f"Tokenizer already present at {tokenizer_pkl}, skipping tok_train.")
else:
    # SMOKE_TEST: train on far fewer characters, just enough to produce a valid tokenizer fast.
    max_chars = 2_000_000 if SMOKE_TEST else 2_000_000_000
    print(f"No tokenizer found, training one on up to {max_chars:,} chars...")
    !python -m scripts.tok_train --max-chars={max_chars}
    if not SMOKE_TEST:
        print("Caching tokenizer to Drive for future sessions...")
        !rclone copy {TOKENIZER_DIR} {TOKENIZER_REMOTE} --checksum -v

Found cached dataset on Drive, downloading instead of re-fetching from HuggingFace...
2026/08/10 18:56:08 INFO  : There was nothing to transfer
2026/08/10 18:56:08 INFO  : 
Transferred:   	          0 B / 0 B, -, 0 B/s, ETA -
Checks:                21 / 21, 100%, Listed 42
Elapsed time:         1.4s

Tokenizer already present at /kaggle/working/nanochat_cache/tokenizer/tokenizer.pkl, skipping tok_train.


## Cell 4: train, with a background watcher syncing new checkpoints to Drive as they're saved

nanochat's `save_checkpoint()` runs in-process inside `base_train.py` and writes straight to local
disk; we don't patch that vendored code to call `rclone` directly. Instead `kaggle/sync_checkpoints.py`
polls the checkpoint dirs every 120s in the background and uploads anything new -- functionally the
same "upload right after every save" behavior, decoupled from nanochat's internals.

In [9]:
import os
import subprocess
import sys

REPO_DIR = "/kaggle/working/repo"
os.chdir(REPO_DIR)

# ~464.0M tokens / 262,144 tokens-per-step ~= 1770 steps this run, so --save-every=100
# gives ~18 checkpoints -- plenty of resume points across sessions without saving too often.
SAVE_EVERY = 5 if SMOKE_TEST else 100

SYNC_LOG = "/kaggle/working/sync_checkpoints.log"
sync_proc = subprocess.Popen(
    [sys.executable, "kaggle/sync_checkpoints.py", "--remote", "gdrive:", "--interval", "30" if SMOKE_TEST else "120", "--log-file", SYNC_LOG],
    env=os.environ.copy(),
)
print(f"Started background checkpoint sync watcher, pid={sync_proc.pid}, log={SYNC_LOG}")

resume_step = int(os.environ.get("RESUME_STEP", "-1"))
resume_args = f"--resume-from-step={resume_step}" if resume_step >= 0 else ""

# T4 has no Flash Attention 3 support -> falls back to PyTorch SDPA, which nanochat itself
# warns has no sliding-window support ("--window-pattern=L" for full-context attention
# instead of the default alternating "SSSL", or GPU utilization is "terrible").
WINDOW_PATTERN = "--window-pattern=L"

if SMOKE_TEST:
    # Tiny, fast run: depth=4 regardless of DEPTH above (this is just a pipeline smoke test),
    # ~20 steps, no CORE eval / sampling, just to prove torchrun + the training loop +
    # checkpoint saving actually work on this GPU. Uses its own model tag (EFFECTIVE_MODEL_TAG,
    # set in Cell 2) so it can't collide with a real run.
    # total_batch_size must be a multiple of device_batch_size * max_seq_len * world_size
    # (= 2 * 512 * 2 = 2048 tokens per forward/backward across both GPUs) -- one grad-accum step.
    train_cmd = (
        "torchrun --standalone --nproc_per_node=2 -m scripts.base_train -- "
        f"--depth=4 {WINDOW_PATTERN} --max-seq-len=512 --device-batch-size=2 --total-batch-size=2048 "
        "--num-iterations=20 --eval-every=10 --eval-tokens=2048 "
        "--core-metric-every=-1 --sample-every=-1 "
        f"--save-every={SAVE_EVERY} {resume_args} --run=dummy --model-tag={EFFECTIVE_MODEL_TAG}"
    )
else:
    # T4 x2: no --fp8 (H100-only feature). --device-batch-size=8 (from DEVICE_BATCH_SIZE,
    # Cell 1) -- the VRAM probe verified depth=6 safely handles up to 13, but 8 is the value
    # that evenly divides the auto-computed total_batch_size (see Cell 1's comment); no reason
    # to fight that just to use a couple more points of headroom.
    # --target-param-data-ratio=20 (Chinchilla-compute-optimal, same as the original d4 run) --
    # see docs/RESEARCH_LOG.md for why this was chosen over overtraining a smaller model.
    train_cmd = (
        "torchrun --standalone --nproc_per_node=2 -m scripts.base_train -- "
        f"--depth={DEPTH} {WINDOW_PATTERN} --device-batch-size={DEVICE_BATCH_SIZE} --target-param-data-ratio=20 "
        f"--save-every={SAVE_EVERY} {resume_args} --run=dummy --model-tag={EFFECTIVE_MODEL_TAG}"
    )
print(f"Running: {train_cmd}")
try:
    !{train_cmd}
finally:
    sync_proc.terminate()
    sync_proc.wait()
    print("Training cell finished (or was interrupted), sync watcher stopped.")

Started background checkpoint sync watcher, pid=678, log=/kaggle/working/sync_checkpoints.log
Running: torchrun --standalone --nproc_per_node=2 -m scripts.base_train -- --depth=6 --window-pattern=L --device-batch-size=8 --target-param-data-ratio=20 --save-every=100  --run=dummy --model-tag=d6
sync_checkpoints: base_dir=/kaggle/working/nanochat_cache remote=gdrive: interval=120s once=False
[2026-08-10 18:56:09] OK   tokenizer -> gdrive:/tokenizer
W0810 18:56:10.334000 679 torch/distributed/run.py:803] 
W0810 18:56:10.334000 679 torch/distributed/run.py:803] *****************************************
W0810 18:56:10.334000 679 torch/distributed/run.py:803] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0810 18:56:10.334000 679 torch/distributed/run.py:803] *****************************************
[2026-08-10 18:56:11] OK   base

## Cell 5: final sync + log summary

Run this even if Cell 4 was interrupted (e.g. by the 12h hard limit) -- it catches anything the
background watcher missed since its last poll, so the session can end safely.

In [10]:
import os
import subprocess
import sys

REPO_DIR = "/kaggle/working/repo"
os.chdir(REPO_DIR)

SYNC_LOG = "/kaggle/working/sync_checkpoints.log"
result = subprocess.run(
    [sys.executable, "kaggle/sync_checkpoints.py", "--remote", "gdrive:", "--once", "--log-file", SYNC_LOG],
    env=os.environ.copy(),
)
print("Final sync exit code:", result.returncode)

if os.path.exists(SYNC_LOG):
    with open(SYNC_LOG) as f:
        lines = f.readlines()
    print("".join(lines[-40:]))

if result.returncode != 0:
    print("WARNING: final sync reported a failure -- check the log above before ending the session.")
else:
    print("All checkpoints synced to Google Drive. Safe to let the session end.")

sync_checkpoints: base_dir=/kaggle/working/nanochat_cache remote=gdrive: interval=120s once=True
[2026-08-11 00:20:32] OK   tokenizer -> gdrive:/tokenizer
[2026-08-11 00:20:45] OK   base_checkpoints -> gdrive:/base_checkpoints
[2026-08-11 00:20:46] OK   chatsft_checkpoints -> gdrive:/chatsft_checkpoints
Final sync exit code: 0
[2026-08-10 23:53:30] OK   tokenizer -> gdrive:/tokenizer
[2026-08-10 23:53:46] OK   base_checkpoints -> gdrive:/base_checkpoints
[2026-08-10 23:53:48] OK   chatsft_checkpoints -> gdrive:/chatsft_checkpoints
[2026-08-10 23:55:48] OK   tokenizer -> gdrive:/tokenizer
[2026-08-10 23:56:01] OK   base_checkpoints -> gdrive:/base_checkpoints
[2026-08-10 23:56:03] OK   chatsft_checkpoints -> gdrive:/chatsft_checkpoints
[2026-08-10 23:58:04] OK   tokenizer -> gdrive:/tokenizer
[2026-08-10 23:58:17] OK   base_checkpoints -> gdrive:/base_checkpoints
[2026-08-10 23:58:18] OK   chatsft_checkpoints -> gdrive:/chatsft_checkpoints
[2026-08-11 00:00:19] OK   tokenizer -> gdrive: